<a href="https://colab.research.google.com/github/shobhakulal79/shobha/blob/main/mini_proj2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ROCK PAPER SCISSORS USING CNN
# Real-time webcam application in Google Colab

!pip install -q tensorflow tensorflow-datasets gradio importlib_resources

import tensorflow as tf
import tensorflow_datasets as tfds
import gradio as gr
import numpy as np
import random

# 1. Load the dataset
print("Loading Rock Paper Scissors dataset...")

(train_data, test_data), info = tfds.load(
    "rock_paper_scissors",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

class_names = info.features["label"].names
print("Classes:", class_names)

# 2. Preprocess images
IMG_SIZE = 150
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_data = (
    train_data
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_data = (
    test_data
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# 3. Build the CNN model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    tf.keras.layers.Conv2D(
        32, (3, 3), activation="relu"
    ),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(
        64, (3, 3), activation="relu"
    ),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(
        128, (3, 3), activation="relu"
    ),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 4. Train the CNN model
print("\nTraining CNN model...")

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=1
)

# 5. Evaluate the model
loss, accuracy = model.evaluate(test_data, verbose=0)

print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

# 6. Predict hand gesture and play the game
def play_game(image):
    if image is None:
        return "Please capture an image of your hand."

    image = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(image, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]
    predicted_index = int(np.argmax(predictions))
    gesture = class_names[predicted_index]
    confidence = float(predictions[predicted_index]) * 100

    computer = random.choice(class_names)

    if gesture == computer:
        result = "DRAW!"
    elif (
        (gesture == "rock" and computer == "scissors") or
        (gesture == "paper" and computer == "rock") or
        (gesture == "scissors" and computer == "paper")
    ):
        result = "YOU WIN!"
    else:
        result = "COMPUTER WINS!"

    return (
        f"   CNN Prediction: {gesture.upper()}\n"
        f"  Confidence: {confidence:.2f}%\n"
        f"   Computer Choice: {computer.upper()}\n"
        f"  Result: {result}"
    )

# 7. Create the application with image upload

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown(
        "# ✊ Rock Paper Scissors ✋\n"
        "### CNN Hand Gesture Recognition\n"
        "Upload an image from your gallery and play!"
    )

    image_input = gr.Image(
        type="pil",
        sources=["upload"],
        label="Upload Rock, Paper or Scissors Image"
    )

    play_button = gr.Button(
        "🔍 Predict & Play",
        variant="primary"
    )

    output = gr.Textbox(
        label="CNN Prediction and Game Result",
        lines=5
    )

    play_button.click(
        fn=play_game,
        inputs=image_input,
        outputs=output
    )

print("Starting the application...")
app.launch(share=True)

Loading Rock Paper Scissors dataset...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

 Trainable params: 4,828,739 (18.42 MB)

 Non-trainable params: 0 (0.00 B)


Training CNN model...
79/79 ━━━━━━━━━━━━━━━━━━━━ 153s 2s/step - accuracy: 0.8198 - loss: 0.4184 - val_accuracy: 0.7446 - val_loss: 0.8096

Test Accuracy: 74.46%


/tmp/ipykernel_1592/2724797750.py:132: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Starting the application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c1155b4fc7881b163.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
